# COMPX525 Assignment 2 - BirdCLEF+ 2026

This notebook is the staged Kaggle workspace for Assignment 2.

Default run mode is intentionally conservative:
- load Kaggle metadata
- verify the 234-label target space
- build deterministic train/validation manifests
- save logs and summary files under `/kaggle/working`

After the first run is clean, enable the optional smoke-test flags in the
configuration cell. Do not start full embedding extraction or model training
until the smoke-test output has been reviewed.



## Run Plan

1. **Stage 0 - EDA and split manifests**
   - No audio loading.
   - Produces `a2_run.log`, `eda_summary.json`,
     `train_audio_split_manifest.csv`, and
     `train_soundscape_split_manifest.csv`.

2. **Stage 1 - BirdNET smoke test**
   - Enable `RUN_BIRDNET_SMOKE = True`.
   - Requires Kaggle internet while installing `birdnet-analyzer`, unless the
     package is attached as an offline Kaggle Dataset.
   - Runs on a small class-balanced subset only.

3. **Stage 2 - Full BirdNET embedding extraction**
   - Enable only after Stage 1 succeeds.
   - Produces a cache that can be reused for the ML classifier experiments.

4. **Stage 2b - BirdNET embedding ML baseline**
   - Attach/download chunk outputs.
   - Train a reproducible non-neural classifier for experiment A1.

5. **Stage 3 - Neural baseline**
   - Enable after the split manifests and audio path checks are clean.
   - First run the PyTorch smoke test.
   - Then train a mel-spectrogram compact CNN baseline.

6. **Stage 4 - Synthetic enrichment**
   - Generate synthetic samples as a preprocessing artifact.
   - Log prompts, seeds, model version, and filter decisions in a manifest.



## Configuration



In [ ]:
from pathlib import Path


SEED = 525

# Stage flags. Keep these false for the first run.
RUN_BIRDNET_SMOKE = False
RUN_BIRDNET_BENCHMARK = False
RUN_BIRDNET_CHUNK = False
RUN_BIRDNET_FULL = False
RUN_BIRDNET_ML_BASELINE = False
RUN_NEURAL_SMOKE = False
RUN_NEURAL_BASELINE = True

# The first BirdNET run should be tiny. Increase only after the CLI and output
# format are confirmed on Kaggle.
SMOKE_MAX_CLASSES = 12
SMOKE_FILES_PER_CLASS = 1
BIRDNET_BENCHMARK_MAX_FILES = 128
BIRDNET_BATCH_SIZE = 16
BIRDNET_THREADS = 2
BIRDNET_CHUNK_INDEX = 0
BIRDNET_NUM_CHUNKS = 8
BIRDNET_CHUNK_SPLIT = "all"  # one of: all, train, val
BIRDNET_CHUNK_MAX_FILES = None
KEEP_RAW_BIRDNET_CSV = False

# Optional directories to scan for previously generated BirdNET chunk outputs.
# On Kaggle, attach chunk notebook outputs as inputs and leave this empty; the
# notebook scans /kaggle/input automatically. Locally, set explicit paths if
# needed.
BIRDNET_FEATURE_INPUT_DIRS = []
ML_CLASSIFIER_MAX_ITER = 15
ML_USE_SECONDARY_LABELS = True

# Neural baseline defaults are intentionally modest for the first validated
# Kaggle run. Increase row caps and epochs after the smoke path is clean.
NEURAL_TRAIN_MAX_ROWS = 12000
NEURAL_VAL_MAX_ROWS = 4096
NEURAL_EPOCHS = 4
NEURAL_BATCH_SIZE = 24
NEURAL_NUM_WORKERS = 2
NEURAL_SECONDS = 5.0
NEURAL_TARGET_SR = 32000
NEURAL_LR = 2e-3
NEURAL_WEIGHT_DECAY = 1e-4
NEURAL_USE_SECONDARY_LABELS = True
NEURAL_INSTALL_TORCH_CU121 = True
NEURAL_FORCE_CPU = False
NEURAL_INCLUDE_SYNTHETIC = True
NEURAL_SYNTHETIC_MAX_ROWS = None

# Validation policy required by the assignment:
# - singleton train_audio classes go to validation
# - other classes get approximately 20% validation, with at least one training
#   example left when possible
VALIDATION_FRACTION = 0.20

# Full embedding extraction can take a long time. This cap is a safety valve
# while debugging. Set to None only after the smoke run succeeds.
FULL_BIRDNET_MAX_FILES = None

# Kaggle competition path candidates. The first existing path with the metadata
# files wins.
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/competitions/birdclef-2026"),
    Path("/kaggle/input/birdclef-2026"),
    Path("/kaggle/input/birdclef-2026-repack"),
    Path("/kaggle/input"),
    Path.cwd() / "assignment_2",
    Path.cwd(),
]

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "a2_working"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"WORK_DIR={WORK_DIR}")
print(f"RUN_BIRDNET_SMOKE={RUN_BIRDNET_SMOKE}")
print(f"RUN_BIRDNET_BENCHMARK={RUN_BIRDNET_BENCHMARK}")
print(f"RUN_BIRDNET_CHUNK={RUN_BIRDNET_CHUNK}")
print(f"RUN_BIRDNET_FULL={RUN_BIRDNET_FULL}")
print(f"RUN_BIRDNET_ML_BASELINE={RUN_BIRDNET_ML_BASELINE}")
print(f"RUN_NEURAL_SMOKE={RUN_NEURAL_SMOKE}")
print(f"RUN_NEURAL_BASELINE={RUN_NEURAL_BASELINE}")



## Imports and Logging



In [ ]:
import ast
import gc
import json
import logging
import math
import os
import random
import shutil
import shlex
import subprocess
import sys
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

warnings.filterwarnings("ignore")


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(SEED)

LOG_PATH = WORK_DIR / "a2_run.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(LOG_PATH, mode="w"),
    ],
)
logger = logging.getLogger("compx525-a2")

logger.info("Python: %s", sys.version.replace("\n", " "))
logger.info("Working directory: %s", WORK_DIR)
logger.info("Seed: %s", SEED)


def log_json(title: str, payload: dict) -> None:
    logger.info("%s:\n%s", title, json.dumps(payload, indent=2, sort_keys=True, default=str))


def run_cmd(cmd: Sequence[str], check: bool = True, cwd: Optional[Path] = None) -> subprocess.CompletedProcess:
    logger.info("CMD: %s", shlex.join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    logger.info("CMD exit=%s", result.returncode)
    if result.stdout:
        logger.info("CMD output:\n%s", result.stdout[-8000:])
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit={result.returncode}: {shlex.join(map(str, cmd))}")
    return result




## Locate Kaggle Data



In [ ]:
REQUIRED_METADATA = [
    "train.csv",
    "taxonomy.csv",
    "sample_submission.csv",
    "train_soundscapes_labels.csv",
]


def list_input_tree(root: Path = Path("/kaggle/input"), max_entries: int = 80) -> List[str]:
    if not root.exists():
        return [f"{root} does not exist"]

    entries = []
    for path in sorted(root.rglob("*")):
        rel = path.relative_to(root)
        suffix = "/" if path.is_dir() else ""
        entries.append(f"{rel}{suffix}")
        if len(entries) >= max_entries:
            entries.append(f"... truncated after {max_entries} entries")
            break
    if not entries:
        entries.append(f"{root} exists but is empty")
    return entries


def candidate_has_metadata(path: Path) -> bool:
    return all((path / name).exists() for name in REQUIRED_METADATA)


def find_data_root() -> Path:
    logger.info("/kaggle/input tree sample:\n%s", "\n".join(list_input_tree()))
    for root in DATA_ROOT_CANDIDATES:
        logger.info("Checking data root candidate: %s exists=%s", root, root.exists())
        if candidate_has_metadata(root):
            return root
        if root.exists():
            for child in sorted(root.glob("*")):
                logger.info("Checking child candidate: %s exists=%s", child, child.exists())
                if child.is_dir() and candidate_has_metadata(child):
                    return child
    raise FileNotFoundError(
        "Could not find BirdCLEF metadata files. Attach the Kaggle competition "
        "dataset, or update DATA_ROOT_CANDIDATES."
    )


DATA_ROOT = find_data_root()
TRAIN_AUDIO_DIR = DATA_ROOT / "train_audio"
TRAIN_SOUNDSCAPES_DIR = DATA_ROOT / "train_soundscapes"
TEST_SOUNDSCAPES_DIR = DATA_ROOT / "test_soundscapes"

logger.info("DATA_ROOT=%s", DATA_ROOT)
logger.info("TRAIN_AUDIO_DIR exists=%s path=%s", TRAIN_AUDIO_DIR.exists(), TRAIN_AUDIO_DIR)
logger.info("TRAIN_SOUNDSCAPES_DIR exists=%s path=%s", TRAIN_SOUNDSCAPES_DIR.exists(), TRAIN_SOUNDSCAPES_DIR)
logger.info("TEST_SOUNDSCAPES_DIR exists=%s path=%s", TEST_SOUNDSCAPES_DIR.exists(), TEST_SOUNDSCAPES_DIR)



## Load Metadata



In [ ]:
train_df = pd.read_csv(DATA_ROOT / "train.csv")
taxonomy_df = pd.read_csv(DATA_ROOT / "taxonomy.csv")
sample_submission_df = pd.read_csv(DATA_ROOT / "sample_submission.csv")
soundscape_labels_df = pd.read_csv(DATA_ROOT / "train_soundscapes_labels.csv")

label_cols = [c for c in sample_submission_df.columns if c != "row_id"]
taxonomy_labels = taxonomy_df["primary_label"].astype(str).tolist()

train_df["primary_label"] = train_df["primary_label"].astype(str)
taxonomy_df["primary_label"] = taxonomy_df["primary_label"].astype(str)
soundscape_labels_df["primary_label"] = soundscape_labels_df["primary_label"].astype(str)

logger.info("train_df shape=%s", train_df.shape)
logger.info("taxonomy_df shape=%s", taxonomy_df.shape)
logger.info("sample_submission_df shape=%s", sample_submission_df.shape)
logger.info("soundscape_labels_df shape=%s", soundscape_labels_df.shape)
logger.info("label_cols=%s taxonomy_labels=%s match=%s", len(label_cols), len(taxonomy_labels), label_cols == taxonomy_labels)

assert label_cols == taxonomy_labels, "sample_submission columns must match taxonomy order"

label_to_idx = {label: i for i, label in enumerate(label_cols)}
idx_to_label = {i: label for label, i in label_to_idx.items()}



## EDA Summary



In [ ]:
def parse_secondary_labels(value) -> List[str]:
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(x) for x in value]
    value = str(value).strip()
    if not value:
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass
    return []


def parse_soundscape_label_string(value) -> List[str]:
    if pd.isna(value):
        return []
    value = str(value).strip()
    if not value or value.lower() == "nocall":
        return []
    return [x.strip() for x in value.split(";") if x.strip()]


train_counts = train_df["primary_label"].value_counts().sort_index()
taxonomy_set = set(label_cols)
train_label_set = set(train_counts.index)

soundscape_label_counter = Counter()
soundscape_segment_label_counts = []
for raw in soundscape_labels_df["primary_label"]:
    labels = parse_soundscape_label_string(raw)
    soundscape_label_counter.update(labels)
    soundscape_segment_label_counts.append(len(labels))

soundscape_label_set = set(soundscape_label_counter.keys())
missing_from_train = sorted(taxonomy_set - train_label_set)
missing_from_train_but_in_soundscapes = sorted((taxonomy_set - train_label_set) & soundscape_label_set)
missing_from_both = sorted(taxonomy_set - train_label_set - soundscape_label_set)
soundscape_not_in_taxonomy = sorted(soundscape_label_set - taxonomy_set)

class_counts_taxonomy = taxonomy_df["class_name"].value_counts().to_dict()
class_counts_train_rows = train_df["class_name"].value_counts().to_dict()

rare_buckets = {
    "n_eq_1": int((train_counts == 1).sum()),
    "n_eq_2": int((train_counts == 2).sum()),
    "n_eq_3": int((train_counts == 3).sum()),
    "n_le_5": int((train_counts <= 5).sum()),
    "n_le_10": int((train_counts <= 10).sum()),
    "n_ge_100": int((train_counts >= 100).sum()),
    "n_ge_400": int((train_counts >= 400).sum()),
}

eda_summary = {
    "data_root": str(DATA_ROOT),
    "train_rows": int(len(train_df)),
    "taxonomy_classes": int(len(label_cols)),
    "train_audio_classes": int(len(train_label_set)),
    "soundscape_labeled_rows": int(len(soundscape_labels_df)),
    "soundscape_labeled_files": int(soundscape_labels_df["filename"].nunique()),
    "taxonomy_class_counts": class_counts_taxonomy,
    "train_row_class_counts": class_counts_train_rows,
    "missing_from_train_count": len(missing_from_train),
    "missing_from_train": missing_from_train,
    "missing_from_train_but_in_soundscapes_count": len(missing_from_train_but_in_soundscapes),
    "missing_from_train_but_in_soundscapes": missing_from_train_but_in_soundscapes,
    "missing_from_both_count": len(missing_from_both),
    "missing_from_both": missing_from_both,
    "soundscape_not_in_taxonomy_count": len(soundscape_not_in_taxonomy),
    "soundscape_not_in_taxonomy": soundscape_not_in_taxonomy,
    "rare_buckets": rare_buckets,
    "top_train_classes": train_counts.sort_values(ascending=False).head(15).to_dict(),
    "rare_train_classes_le_5": train_counts[train_counts <= 5].sort_values().to_dict(),
    "top_soundscape_labels": dict(soundscape_label_counter.most_common(15)),
    "soundscape_labels_per_segment_min": int(min(soundscape_segment_label_counts)),
    "soundscape_labels_per_segment_median": float(np.median(soundscape_segment_label_counts)),
    "soundscape_labels_per_segment_max": int(max(soundscape_segment_label_counts)),
}

log_json("EDA summary", eda_summary)

(WORK_DIR / "eda_summary.json").write_text(json.dumps(eda_summary, indent=2, sort_keys=True), encoding="utf-8")
train_counts.rename("count").to_csv(WORK_DIR / "train_audio_class_counts.csv")
taxonomy_df.to_csv(WORK_DIR / "taxonomy_checked.csv", index=False)

display(pd.DataFrame([eda_summary]).T.rename(columns={0: "value"}).head(40))
display(train_counts.sort_values(ascending=False).head(20).rename("count").reset_index().rename(columns={"index": "primary_label"}))
display(train_counts[train_counts <= 5].sort_values().rename("count").reset_index().rename(columns={"index": "primary_label"}))



## Deterministic Train/Validation Splits

The report should state this policy explicitly:
- all `train.csv` classes with one example are forced into validation
- all other `train.csv` classes retain at least one training example
- soundscape labeled segments are split by filename, not by segment, to avoid
  adjacent-segment leakage



In [ ]:
def make_train_audio_split(df: pd.DataFrame, val_fraction: float, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    pieces = []
    for label, group in df.groupby("primary_label", sort=True):
        group = group.copy()
        indices = np.array(group.index)
        rng.shuffle(indices)
        n = len(indices)

        if n == 1:
            val_indices = set(indices.tolist())
        else:
            val_n = max(1, int(round(n * val_fraction)))
            val_n = min(val_n, n - 1)
            val_indices = set(indices[:val_n].tolist())

        group["split"] = ["val" if idx in val_indices else "train" for idx in group.index]
        group["split_reason"] = "singleton_forced_val" if n == 1 else "stratified_by_primary_label"
        pieces.append(group)

    split_df = pd.concat(pieces, axis=0).sort_index()
    return split_df


def make_soundscape_file_split(df: pd.DataFrame, val_fraction: float, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    files = np.array(sorted(df["filename"].unique()))
    rng.shuffle(files)
    val_n = max(1, int(round(len(files) * val_fraction)))
    val_files = set(files[:val_n].tolist())
    out = df.copy()
    out["split"] = out["filename"].map(lambda name: "val" if name in val_files else "train")
    out["split_reason"] = "file_level_random_split"
    out["labels_list"] = out["primary_label"].map(lambda x: ";".join(parse_soundscape_label_string(x)))
    return out


train_audio_split_df = make_train_audio_split(train_df, VALIDATION_FRACTION, SEED)
soundscape_split_df = make_soundscape_file_split(soundscape_labels_df, VALIDATION_FRACTION, SEED)

train_audio_split_path = WORK_DIR / "train_audio_split_manifest.csv"
soundscape_split_path = WORK_DIR / "train_soundscape_split_manifest.csv"
train_audio_split_df.to_csv(train_audio_split_path, index=False)
soundscape_split_df.to_csv(soundscape_split_path, index=False)

split_summary = {
    "train_audio_rows_by_split": train_audio_split_df["split"].value_counts().to_dict(),
    "train_audio_classes_in_train": int(train_audio_split_df.loc[train_audio_split_df["split"] == "train", "primary_label"].nunique()),
    "train_audio_classes_in_val": int(train_audio_split_df.loc[train_audio_split_df["split"] == "val", "primary_label"].nunique()),
    "train_audio_singleton_val_rows": int((train_audio_split_df["split_reason"] == "singleton_forced_val").sum()),
    "soundscape_rows_by_split": soundscape_split_df["split"].value_counts().to_dict(),
    "soundscape_files_by_split": soundscape_split_df.groupby("split")["filename"].nunique().to_dict(),
    "manifest_paths": {
        "train_audio": str(train_audio_split_path),
        "train_soundscapes": str(soundscape_split_path),
    },
}
log_json("Split summary", split_summary)
(WORK_DIR / "split_summary.json").write_text(json.dumps(split_summary, indent=2, sort_keys=True), encoding="utf-8")

display(pd.DataFrame([split_summary]).T.rename(columns={0: "value"}))
display(train_audio_split_df.groupby(["split", "split_reason"]).size().rename("rows").reset_index())
display(soundscape_split_df.groupby("split")["filename"].nunique().rename("files").reset_index())



## Target Encoding and Metric



In [ ]:
def encode_single_label(label: str, label_to_idx: Dict[str, int]) -> np.ndarray:
    y = np.zeros(len(label_to_idx), dtype=np.float32)
    y[label_to_idx[str(label)]] = 1.0
    return y


def encode_multi_label(labels: Sequence[str], label_to_idx: Dict[str, int]) -> np.ndarray:
    y = np.zeros(len(label_to_idx), dtype=np.float32)
    for label in labels:
        if label in label_to_idx:
            y[label_to_idx[label]] = 1.0
    return y


def kaggle_macro_auc_skip_empty(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Kaggle BirdCLEF metric: macro ROC-AUC over classes with positives."""
    try:
        from sklearn.metrics import roc_auc_score
    except Exception as exc:
        logger.warning("sklearn is unavailable; metric calculation skipped: %s", repr(exc))
        return float("nan")

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    scored = y_true.sum(axis=0) > 0
    if scored.sum() == 0:
        raise ValueError("No positive classes in y_true; cannot score.")
    return float(roc_auc_score(y_true[:, scored], y_pred[:, scored], average="macro"))


# Smoke check metric with random predictions. This only verifies dimensions.
val_labels = train_audio_split_df.loc[train_audio_split_df["split"] == "val", "primary_label"].tolist()
y_true_smoke = np.stack([encode_single_label(label, label_to_idx) for label in val_labels])
rng = np.random.default_rng(SEED)
y_pred_smoke = rng.random(y_true_smoke.shape)
metric_smoke_value = kaggle_macro_auc_skip_empty(y_true_smoke, y_pred_smoke)
logger.info("Metric smoke value with random predictions: %.6f", metric_smoke_value)



## Audio Path Checks

This section checks for file existence only. It does not read audio samples.



In [ ]:
def train_audio_path(filename: str) -> Path:
    return TRAIN_AUDIO_DIR / str(filename)


def train_soundscape_path(filename: str) -> Path:
    return TRAIN_SOUNDSCAPES_DIR / str(filename)


def check_paths_exist(paths: Sequence[Path], max_missing: int = 10) -> Tuple[int, List[str]]:
    missing = [str(p) for p in paths if not p.exists()]
    return len(missing), missing[:max_missing]


sample_train_paths = [train_audio_path(x) for x in train_df["filename"].head(200)]
sample_soundscape_paths = [train_soundscape_path(x) for x in soundscape_labels_df["filename"].drop_duplicates().head(50)]

train_missing_count, train_missing_examples = check_paths_exist(sample_train_paths)
soundscape_missing_count, soundscape_missing_examples = check_paths_exist(sample_soundscape_paths)

audio_path_summary = {
    "checked_train_audio_paths": len(sample_train_paths),
    "missing_train_audio_count": train_missing_count,
    "missing_train_audio_examples": train_missing_examples,
    "checked_train_soundscape_paths": len(sample_soundscape_paths),
    "missing_train_soundscape_count": soundscape_missing_count,
    "missing_train_soundscape_examples": soundscape_missing_examples,
}
log_json("Audio path summary", audio_path_summary)
(WORK_DIR / "audio_path_summary.json").write_text(json.dumps(audio_path_summary, indent=2, sort_keys=True), encoding="utf-8")
display(pd.DataFrame([audio_path_summary]).T.rename(columns={0: "value"}))



## Stage 1 Optional: BirdNET Analyzer Smoke Test

Enable `RUN_BIRDNET_SMOKE = True` in the configuration cell after Stage 0 is
clean. If installation fails, turn on Kaggle internet or attach BirdNET as a
Kaggle Dataset.



In [ ]:
def ensure_birdnet_analyzer() -> bool:
    required_imports = ["birdnet_analyzer", "perch_hoplite"]
    try:
        import birdnet_analyzer  # noqa: F401
        import perch_hoplite  # noqa: F401

        logger.info("birdnet_analyzer and perch_hoplite are already installed")
        return True
    except Exception as exc:
        logger.info("BirdNET import check failed: %s", repr(exc))

    logger.info("Attempting pip install birdnet-analyzer perch-hoplite==0.1.3")
    result = run_cmd(
        [sys.executable, "-m", "pip", "install", "-q", "birdnet-analyzer", "perch-hoplite==0.1.3"],
        check=False,
    )
    if result.returncode != 0:
        logger.error("BirdNET dependency install failed. Check Kaggle internet/offline package setup.")
        return False

    try:
        import birdnet_analyzer  # noqa: F401
        import perch_hoplite  # noqa: F401

        logger.info("BirdNET dependencies installed successfully: %s", required_imports)
        return True
    except Exception as exc:
        logger.error("BirdNET dependencies still cannot be imported: %s", repr(exc))
        return False


def make_smoke_audio_tree(split_df: pd.DataFrame, output_root: Path) -> pd.DataFrame:
    """Create a tiny class-balanced symlink tree for BirdNET smoke extraction."""
    if output_root.exists():
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    selected = []
    class_order = split_df["primary_label"].value_counts().sort_values(ascending=True).index.tolist()
    for label in class_order[:SMOKE_MAX_CLASSES]:
        rows = split_df[(split_df["primary_label"] == label) & (split_df["split"] == "train")].head(SMOKE_FILES_PER_CLASS)
        if rows.empty:
            rows = split_df[split_df["primary_label"] == label].head(SMOKE_FILES_PER_CLASS)
        selected.append(rows)

    selected_df = pd.concat(selected, axis=0).drop_duplicates("filename").copy()

    created = []
    for _, row in selected_df.iterrows():
        source = train_audio_path(row["filename"])
        target_dir = output_root / str(row["primary_label"])
        target_dir.mkdir(parents=True, exist_ok=True)
        target = target_dir / Path(row["filename"]).name
        if not source.exists():
            logger.warning("Missing smoke source audio: %s", source)
            continue
        try:
            os.symlink(source, target)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(source, target)
        created.append(str(target))

    selected_df["smoke_audio_path"] = created + [None] * max(0, len(selected_df) - len(created))
    logger.info("Smoke audio files prepared: %s", len(created))
    return selected_df


def make_limited_audio_tree(split_df: pd.DataFrame, output_root: Path, max_files: int) -> pd.DataFrame:
    """Create a deterministic train-only symlink tree for BirdNET benchmarking."""
    if output_root.exists():
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    train_rows = split_df[split_df["split"] == "train"].copy()
    balanced = train_rows.groupby("primary_label", group_keys=False).head(1)
    remainder = train_rows.drop(index=balanced.index, errors="ignore")
    selected_df = pd.concat([balanced, remainder], axis=0).head(max_files).copy()

    created = []
    for _, row in selected_df.iterrows():
        source = train_audio_path(row["filename"])
        target_dir = output_root / str(row["primary_label"])
        target_dir.mkdir(parents=True, exist_ok=True)
        target = target_dir / Path(row["filename"]).name
        if not source.exists():
            logger.warning("Missing benchmark source audio: %s", source)
            continue
        try:
            os.symlink(source, target)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(source, target)
        created.append(str(target))

    selected_df = selected_df.iloc[: len(created)].copy()
    selected_df["benchmark_audio_path"] = created
    logger.info("Benchmark audio files prepared: %s", len(created))
    return selected_df


def make_audio_tree_from_rows(rows: pd.DataFrame, output_root: Path, path_column: str) -> pd.DataFrame:
    """Create a symlink tree preserving primary_label folders for a selected manifest."""
    if output_root.exists():
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    rows = rows.copy().reset_index(drop=True)
    created = []
    kept_indices = []
    for idx, row in rows.iterrows():
        source = train_audio_path(row["filename"])
        target_dir = output_root / str(row["primary_label"])
        target_dir.mkdir(parents=True, exist_ok=True)
        target = target_dir / Path(row["filename"]).name
        if not source.exists():
            logger.warning("Missing selected source audio: %s", source)
            continue
        try:
            os.symlink(source, target)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(source, target)
        kept_indices.append(idx)
        created.append(str(target))

    out = rows.iloc[kept_indices].copy().reset_index(drop=True)
    out[path_column] = created
    logger.info("Audio files prepared for %s: %s", path_column, len(created))
    return out


def select_birdnet_chunk(split_df: pd.DataFrame) -> pd.DataFrame:
    if BIRDNET_CHUNK_SPLIT not in {"all", "train", "val"}:
        raise ValueError(f"Invalid BIRDNET_CHUNK_SPLIT={BIRDNET_CHUNK_SPLIT}")
    if not 0 <= BIRDNET_CHUNK_INDEX < BIRDNET_NUM_CHUNKS:
        raise ValueError("BIRDNET_CHUNK_INDEX must be in [0, BIRDNET_NUM_CHUNKS)")

    rows = split_df.copy()
    if BIRDNET_CHUNK_SPLIT != "all":
        rows = rows[rows["split"] == BIRDNET_CHUNK_SPLIT].copy()
    rows = rows.sort_values(["primary_label", "filename"]).reset_index(drop=True)
    if BIRDNET_CHUNK_MAX_FILES is not None:
        rows = rows.head(int(BIRDNET_CHUNK_MAX_FILES)).copy()

    chunks = np.array_split(np.arange(len(rows)), BIRDNET_NUM_CHUNKS)
    selected_indices = chunks[BIRDNET_CHUNK_INDEX]
    selected = rows.iloc[selected_indices].copy().reset_index(drop=True)
    logger.info(
        "Selected BirdNET chunk index=%s/%s split=%s rows=%s from_total=%s max_files=%s",
        BIRDNET_CHUNK_INDEX,
        BIRDNET_NUM_CHUNKS,
        BIRDNET_CHUNK_SPLIT,
        len(selected),
        len(rows),
        BIRDNET_CHUNK_MAX_FILES,
    )
    return selected


def run_birdnet_embeddings(input_root: Path, database_root: Path, output_csv: Path) -> subprocess.CompletedProcess:
    if database_root.exists():
        shutil.rmtree(database_root)
    cmd = [
        sys.executable,
        "-m",
        "birdnet_analyzer.embeddings",
        "-db",
        database_root,
        "-i",
        input_root,
        "--file_output",
        output_csv,
        "--batch_size",
        str(BIRDNET_BATCH_SIZE),
        "--threads",
        str(BIRDNET_THREADS),
    ]
    start = time.time()
    result = run_cmd(cmd, check=False)
    elapsed = time.time() - start
    logger.info("BirdNET embeddings elapsed_seconds=%.3f output_exists=%s", elapsed, output_csv.exists())
    return result


def parse_embedding_string(value: str) -> np.ndarray:
    return np.fromstring(str(value), sep=",", dtype=np.float32)


def aggregate_birdnet_embeddings(embedding_csv: Path, output_prefix: Path) -> Optional[dict]:
    if not embedding_csv.exists():
        logger.warning("Cannot aggregate missing embedding CSV: %s", embedding_csv)
        return None

    emb_df = pd.read_csv(embedding_csv)
    if emb_df.empty:
        logger.warning("Embedding CSV is empty: %s", embedding_csv)
        return None

    emb_df["primary_label"] = emb_df["file_path"].map(lambda x: Path(str(x)).parent.name)
    emb_df["audio_basename"] = emb_df["file_path"].map(lambda x: Path(str(x)).name)
    emb_df["filename"] = emb_df["primary_label"] + "/" + emb_df["audio_basename"]

    vectors = np.stack([parse_embedding_string(x) for x in emb_df["embedding"]])
    segment_meta = emb_df[["filename", "file_path", "start", "end", "primary_label"]].copy()
    segment_meta_path = output_prefix.with_name(output_prefix.name + "_segments.csv")
    segment_meta.to_csv(segment_meta_path, index=False)

    filenames = []
    labels = []
    features = []
    for filename, idx in emb_df.groupby("filename").indices.items():
        group_vectors = vectors[np.array(idx)]
        filenames.append(filename)
        labels.append(str(filename).split("/", 1)[0])
        features.append(
            np.concatenate(
                [
                    group_vectors.mean(axis=0),
                    group_vectors.std(axis=0),
                    group_vectors.max(axis=0),
                ]
            )
        )

    feature_array = np.stack(features).astype(np.float32)
    metadata = pd.DataFrame({"filename": filenames, "primary_label": labels})
    metadata_path = output_prefix.with_name(output_prefix.name + "_clip_metadata.csv")
    feature_path = output_prefix.with_name(output_prefix.name + "_clip_features.npz")
    metadata.to_csv(metadata_path, index=False)
    np.savez_compressed(feature_path, features=feature_array, filenames=np.array(filenames), labels=np.array(labels))

    summary = {
        "embedding_csv": str(embedding_csv),
        "segments": int(len(emb_df)),
        "clips": int(len(metadata)),
        "embedding_dim": int(vectors.shape[1]),
        "clip_feature_dim": int(feature_array.shape[1]),
        "segment_meta_path": str(segment_meta_path),
        "metadata_path": str(metadata_path),
        "feature_path": str(feature_path),
    }
    log_json("BirdNET aggregation summary", summary)
    output_prefix.with_name(output_prefix.name + "_summary.json").write_text(
        json.dumps(summary, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    return summary


if RUN_BIRDNET_SMOKE:
    smoke_root = WORK_DIR / "birdnet_smoke_audio"
    smoke_df = make_smoke_audio_tree(train_audio_split_df, smoke_root)
    smoke_df.to_csv(WORK_DIR / "birdnet_smoke_manifest.csv", index=False)
    display(smoke_df[["primary_label", "filename", "split", "smoke_audio_path"]])

    if ensure_birdnet_analyzer():
        run_cmd([sys.executable, "-m", "birdnet_analyzer.embeddings", "--help"], check=False)

        birdnet_db = WORK_DIR / "birdnet_smoke_db"
        birdnet_out = WORK_DIR / "birdnet_smoke_embeddings.csv"
        run_birdnet_embeddings(smoke_root, birdnet_db, birdnet_out)
        logger.info("BirdNET smoke output exists=%s path=%s", birdnet_out.exists(), birdnet_out)
        if birdnet_out.exists():
            logger.info("BirdNET smoke output size=%s bytes", birdnet_out.stat().st_size)
            try:
                birdnet_smoke_df = pd.read_csv(birdnet_out)
                logger.info("BirdNET smoke dataframe shape=%s", birdnet_smoke_df.shape)
                display(birdnet_smoke_df.head())
            except Exception as exc:
                logger.exception("Could not read BirdNET smoke output as CSV: %s", exc)
            aggregate_birdnet_embeddings(birdnet_out, WORK_DIR / "birdnet_smoke")
else:
    logger.info("Skipping BirdNET smoke test. Set RUN_BIRDNET_SMOKE=True to enable.")


if RUN_BIRDNET_BENCHMARK:
    benchmark_root = WORK_DIR / "birdnet_benchmark_audio"
    benchmark_df = make_limited_audio_tree(train_audio_split_df, benchmark_root, BIRDNET_BENCHMARK_MAX_FILES)
    benchmark_df.to_csv(WORK_DIR / "birdnet_benchmark_manifest.csv", index=False)
    display(benchmark_df[["primary_label", "filename", "split", "benchmark_audio_path"]].head(30))

    if ensure_birdnet_analyzer():
        birdnet_db = WORK_DIR / "birdnet_benchmark_db"
        birdnet_out = WORK_DIR / "birdnet_benchmark_embeddings.csv"
        start = time.time()
        result = run_birdnet_embeddings(benchmark_root, birdnet_db, birdnet_out)
        elapsed = time.time() - start
        benchmark_summary = {
            "max_files_requested": BIRDNET_BENCHMARK_MAX_FILES,
            "files_prepared": int(len(benchmark_df)),
            "exit_code": int(result.returncode),
            "elapsed_seconds": float(elapsed),
            "seconds_per_file": float(elapsed / max(1, len(benchmark_df))),
            "output_exists": bool(birdnet_out.exists()),
            "output_size_bytes": int(birdnet_out.stat().st_size) if birdnet_out.exists() else 0,
        }
        if birdnet_out.exists():
            emb_rows = pd.read_csv(birdnet_out, usecols=["file_path"]).shape[0]
            benchmark_summary["embedding_segments"] = int(emb_rows)
            benchmark_summary["seconds_per_segment"] = float(elapsed / max(1, emb_rows))
            aggregate_birdnet_embeddings(birdnet_out, WORK_DIR / "birdnet_benchmark")
        log_json("BirdNET benchmark summary", benchmark_summary)
        (WORK_DIR / "birdnet_benchmark_summary.json").write_text(
            json.dumps(benchmark_summary, indent=2, sort_keys=True),
            encoding="utf-8",
        )
else:
    logger.info("Skipping BirdNET benchmark. Set RUN_BIRDNET_BENCHMARK=True to enable.")


if RUN_BIRDNET_CHUNK:
    chunk_rows = select_birdnet_chunk(train_audio_split_df)
    chunk_name = f"birdnet_{BIRDNET_CHUNK_SPLIT}_chunk{BIRDNET_CHUNK_INDEX:02d}_of{BIRDNET_NUM_CHUNKS:02d}"
    chunk_root = WORK_DIR / f"{chunk_name}_audio"
    chunk_df = make_audio_tree_from_rows(chunk_rows, chunk_root, "chunk_audio_path")
    chunk_manifest_path = WORK_DIR / f"{chunk_name}_manifest.csv"
    chunk_df.to_csv(chunk_manifest_path, index=False)
    display(chunk_df[["primary_label", "filename", "split", "chunk_audio_path"]].head(30))

    if ensure_birdnet_analyzer():
        birdnet_db = WORK_DIR / f"{chunk_name}_db"
        birdnet_out = WORK_DIR / f"{chunk_name}_embeddings.csv"
        start = time.time()
        result = run_birdnet_embeddings(chunk_root, birdnet_db, birdnet_out)
        elapsed = time.time() - start
        aggregation = aggregate_birdnet_embeddings(birdnet_out, WORK_DIR / chunk_name) if birdnet_out.exists() else None

        chunk_summary = {
            "chunk_name": chunk_name,
            "split": BIRDNET_CHUNK_SPLIT,
            "chunk_index": BIRDNET_CHUNK_INDEX,
            "num_chunks": BIRDNET_NUM_CHUNKS,
            "max_files": BIRDNET_CHUNK_MAX_FILES,
            "files_prepared": int(len(chunk_df)),
            "exit_code": int(result.returncode),
            "elapsed_seconds": float(elapsed),
            "seconds_per_file": float(elapsed / max(1, len(chunk_df))),
            "output_exists": bool(birdnet_out.exists()),
            "output_size_bytes": int(birdnet_out.stat().st_size) if birdnet_out.exists() else 0,
            "aggregation": aggregation,
            "manifest_path": str(chunk_manifest_path),
        }
        if aggregation:
            chunk_summary["embedding_segments"] = int(aggregation["segments"])
            chunk_summary["seconds_per_segment"] = float(elapsed / max(1, aggregation["segments"]))

        if birdnet_out.exists() and not KEEP_RAW_BIRDNET_CSV:
            birdnet_out.unlink()
            chunk_summary["raw_embedding_csv_deleted"] = True
        if birdnet_db.exists() and not KEEP_RAW_BIRDNET_CSV:
            shutil.rmtree(birdnet_db)
            chunk_summary["birdnet_db_deleted"] = True

        log_json("BirdNET chunk summary", chunk_summary)
        (WORK_DIR / f"{chunk_name}_summary.json").write_text(
            json.dumps(chunk_summary, indent=2, sort_keys=True),
            encoding="utf-8",
        )
else:
    logger.info("Skipping BirdNET chunk extraction. Set RUN_BIRDNET_CHUNK=True to enable.")



## Stage 2 Optional: Full BirdNET Embedding Extraction

Do not enable this until the smoke test confirms the CLI command and output
format on Kaggle.



In [ ]:
if RUN_BIRDNET_FULL:
    if FULL_BIRDNET_MAX_FILES is not None:
        logger.warning("FULL_BIRDNET_MAX_FILES=%s is set; this is not a full run.", FULL_BIRDNET_MAX_FILES)

    if ensure_birdnet_analyzer():
        full_root = TRAIN_AUDIO_DIR
        if FULL_BIRDNET_MAX_FILES is not None:
            limited_root = WORK_DIR / "birdnet_limited_audio"
            if limited_root.exists():
                shutil.rmtree(limited_root)
            limited_root.mkdir(parents=True, exist_ok=True)
            limited = train_audio_split_df.head(FULL_BIRDNET_MAX_FILES)
            for _, row in limited.iterrows():
                source = train_audio_path(row["filename"])
                target_dir = limited_root / str(row["primary_label"])
                target_dir.mkdir(parents=True, exist_ok=True)
                target = target_dir / Path(row["filename"]).name
                if source.exists():
                    try:
                        os.symlink(source, target)
                    except OSError:
                        shutil.copy2(source, target)
            full_root = limited_root

        birdnet_db = WORK_DIR / "birdnet_full_db"
        birdnet_out = WORK_DIR / "birdnet_train_audio_embeddings.csv"
        cmd = [
            sys.executable,
            "-m",
            "birdnet_analyzer.embeddings",
            "-db",
            birdnet_db,
            "-i",
            full_root,
            "--file_output",
            birdnet_out,
            "--batch_size",
            "16",
        ]
        run_cmd(cmd, check=False)
        logger.info("Full BirdNET output exists=%s path=%s", birdnet_out.exists(), birdnet_out)
else:
    logger.info("Skipping full BirdNET extraction. Set RUN_BIRDNET_FULL=True to enable.")



## Stage 2b Optional: BirdNET Embedding ML Baseline

This trains the non-neural experiment on cached BirdNET clip features. Use it
after chunk outputs are available under `/kaggle/input` or listed in
`BIRDNET_FEATURE_INPUT_DIRS`.



In [ ]:
def discover_birdnet_feature_files(extra_dirs: Sequence[str]) -> List[Path]:
    roots = [WORK_DIR]
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.extend(Path(p) for p in extra_dirs)

    feature_files = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob("birdnet_*_chunk*_clip_features.npz")):
            if path not in seen:
                feature_files.append(path)
                seen.add(path)

    logger.info("Discovered %s BirdNET chunk feature files", len(feature_files))
    for path in feature_files[:30]:
        logger.info("Feature file: %s", path)
    if len(feature_files) > 30:
        logger.info("... %s additional feature files omitted from log", len(feature_files) - 30)
    return feature_files


def load_birdnet_feature_files(feature_files: Sequence[Path]) -> Tuple[np.ndarray, pd.DataFrame]:
    feature_blocks = []
    metadata_blocks = []
    for path in feature_files:
        with np.load(path, allow_pickle=True) as data:
            features = data["features"].astype(np.float32)
            filenames = data["filenames"].astype(str)
            if "labels" in data:
                labels = data["labels"].astype(str)
            else:
                labels = np.array([str(x).split("/", 1)[0] for x in filenames])
        feature_blocks.append(features)
        metadata_blocks.append(
            pd.DataFrame(
                {
                    "filename": filenames,
                    "feature_primary_label": labels,
                    "feature_file": str(path),
                }
            )
        )

    if not feature_blocks:
        raise ValueError("No BirdNET feature files were provided")

    features = np.concatenate(feature_blocks, axis=0)
    metadata = pd.concat(metadata_blocks, axis=0, ignore_index=True)
    keep = ~metadata["filename"].duplicated(keep="first").to_numpy()
    dropped = int((~keep).sum())
    if dropped:
        logger.warning("Dropping duplicated feature rows by filename: %s", dropped)
    return features[keep], metadata.loc[keep].reset_index(drop=True)


def row_labels(row: pd.Series) -> List[str]:
    labels = [str(row["primary_label"])]
    if ML_USE_SECONDARY_LABELS and "secondary_labels" in row:
        labels.extend(parse_secondary_labels(row["secondary_labels"]))
    return sorted(set(label for label in labels if label in label_to_idx))


def train_birdnet_ml_baseline(features: np.ndarray, metadata: pd.DataFrame) -> dict:
    from sklearn.linear_model import SGDClassifier
    from sklearn.multiclass import OneVsRestClassifier
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    join_cols = ["filename", "primary_label", "secondary_labels", "split"]
    joined = metadata.merge(train_audio_split_df[join_cols], how="left", on="filename")
    missing_split = int(joined["split"].isna().sum())
    if missing_split:
        logger.warning("Dropping feature rows not found in train_audio split manifest: %s", missing_split)
    keep = joined["split"].notna().to_numpy()
    joined = joined.loc[keep].reset_index(drop=True)
    features = features[keep]

    labels = [row_labels(row) for _, row in joined.iterrows()]
    y = np.stack([encode_multi_label(items, label_to_idx) for items in labels]).astype(np.float32)
    train_mask = joined["split"].eq("train").to_numpy()
    val_mask = joined["split"].eq("val").to_numpy()

    x_train = features[train_mask]
    y_train = y[train_mask]
    x_val = features[val_mask]
    y_val = y[val_mask]
    if len(x_train) == 0 or len(x_val) == 0:
        raise ValueError(f"Need both train and val features; got train={len(x_train)} val={len(x_val)}")

    logger.info(
        "BirdNET ML baseline train_shape=%s val_shape=%s train_positive_classes=%s val_positive_classes=%s",
        x_train.shape,
        x_val.shape,
        int((y_train.sum(axis=0) > 0).sum()),
        int((y_val.sum(axis=0) > 0).sum()),
    )

    classifier = make_pipeline(
        StandardScaler(),
        OneVsRestClassifier(
            SGDClassifier(
                loss="log_loss",
                alpha=1e-4,
                max_iter=ML_CLASSIFIER_MAX_ITER,
                tol=1e-3,
                class_weight="balanced",
                random_state=SEED,
            ),
            n_jobs=-1,
        ),
    )
    start = time.time()
    classifier.fit(x_train, y_train)
    elapsed = time.time() - start
    val_pred = classifier.predict_proba(x_val).astype(np.float32)
    val_auc = kaggle_macro_auc_skip_empty(y_val, val_pred)

    val_pred_df = pd.DataFrame(val_pred, columns=label_cols)
    val_pred_df.insert(0, "filename", joined.loc[val_mask, "filename"].to_numpy())
    val_pred_path = WORK_DIR / "birdnet_ml_val_predictions.csv"
    val_pred_df.to_csv(val_pred_path, index=False)

    summary = {
        "feature_rows_joined": int(len(joined)),
        "train_rows": int(len(x_train)),
        "val_rows": int(len(x_val)),
        "feature_dim": int(features.shape[1]),
        "labels": int(len(label_cols)),
        "ml_use_secondary_labels": bool(ML_USE_SECONDARY_LABELS),
        "classifier": "StandardScaler + OneVsRestClassifier(SGDClassifier log_loss)",
        "classifier_max_iter": int(ML_CLASSIFIER_MAX_ITER),
        "fit_elapsed_seconds": float(elapsed),
        "val_macro_auc_skip_empty": float(val_auc),
        "train_positive_classes": int((y_train.sum(axis=0) > 0).sum()),
        "val_positive_classes": int((y_val.sum(axis=0) > 0).sum()),
        "train_missing_positive_classes": [
            label_cols[i] for i, value in enumerate(y_train.sum(axis=0)) if value == 0
        ],
        "val_prediction_path": str(val_pred_path),
    }
    log_json("BirdNET ML baseline summary", summary)
    (WORK_DIR / "birdnet_ml_baseline_summary.json").write_text(
        json.dumps(summary, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    return summary


if RUN_BIRDNET_ML_BASELINE:
    feature_files = discover_birdnet_feature_files(BIRDNET_FEATURE_INPUT_DIRS)
    if not feature_files:
        logger.warning("No BirdNET chunk features found; attach chunk outputs or set BIRDNET_FEATURE_INPUT_DIRS.")
    else:
        feature_array, feature_metadata = load_birdnet_feature_files(feature_files)
        feature_metadata.to_csv(WORK_DIR / "birdnet_ml_feature_metadata.csv", index=False)
        ml_summary = train_birdnet_ml_baseline(feature_array, feature_metadata)
        display(pd.DataFrame([ml_summary]).T.rename(columns={0: "value"}))
else:
    logger.info("Skipping BirdNET ML baseline. Set RUN_BIRDNET_ML_BASELINE=True to enable.")



## Stage 3 Optional: Neural Smoke Test

This is a small sanity check for audio loading and mel-spectrogram creation.
It is not the final model. Keep it tiny until the data path and memory behavior
are verified on Kaggle.



In [ ]:
def ensure_neural_torch_runtime() -> None:
    if not NEURAL_INSTALL_TORCH_CU121:
        return
    logger.info("Installing PyTorch/torchaudio cu121 runtime for Kaggle GPU compatibility")
    run_cmd(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "torch==2.5.1+cu121",
            "torchaudio==2.5.1+cu121",
            "--index-url",
            "https://download.pytorch.org/whl/cu121",
        ],
        check=False,
    )


def choose_torch_device(torch_module, nn_module):
    if NEURAL_FORCE_CPU:
        logger.warning("NEURAL_FORCE_CPU=True; using CPU for neural run")
        return torch_module.device("cpu")
    if not torch_module.cuda.is_available():
        logger.warning("CUDA is unavailable; using CPU for neural run")
        return torch_module.device("cpu")
    try:
        test_model = nn_module.Conv2d(1, 1, kernel_size=3, padding=1).cuda()
        test_input = torch_module.randn(1, 1, 8, 8, device="cuda")
        _ = test_model(test_input)
        torch_module.cuda.synchronize()
        logger.info("CUDA conv self-test passed; using GPU")
        return torch_module.device("cuda")
    except Exception as exc:
        logger.warning("CUDA conv self-test failed; falling back to CPU: %s", repr(exc))
        try:
            torch_module.cuda.empty_cache()
        except Exception:
            pass
        return torch_module.device("cpu")


if RUN_NEURAL_SMOKE:
    ensure_neural_torch_runtime()
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset

    try:
        import torchaudio
    except Exception as exc:
        raise RuntimeError("torchaudio is required for this smoke test") from exc

    logger.info("torch=%s cuda_available=%s", torch.__version__, torch.cuda.is_available())
    device = choose_torch_device(torch, nn)

    smoke_train = train_audio_split_df[train_audio_split_df["split"] == "train"].groupby("primary_label").head(1).head(64)
    smoke_val = train_audio_split_df[train_audio_split_df["split"] == "val"].groupby("primary_label").head(1).head(64)

    class AudioSmokeDataset(Dataset):
        def __init__(self, rows: pd.DataFrame, seconds: float = 5.0, target_sr: int = 32000):
            self.rows = rows.reset_index(drop=True)
            self.seconds = seconds
            self.target_sr = target_sr
            self.max_len = int(seconds * target_sr)
            self.mel = torchaudio.transforms.MelSpectrogram(
                sample_rate=target_sr,
                n_fft=1024,
                hop_length=320,
                n_mels=128,
                f_min=20,
                f_max=15000,
                power=2.0,
            )
            self.db = torchaudio.transforms.AmplitudeToDB(stype="power")

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, idx):
            row = self.rows.iloc[idx]
            path = train_audio_path(row["filename"])
            wav, sr = torchaudio.load(str(path))
            wav = wav.mean(dim=0, keepdim=True)
            if sr != self.target_sr:
                wav = torchaudio.functional.resample(wav, sr, self.target_sr)
            if wav.shape[1] < self.max_len:
                wav = torch.nn.functional.pad(wav, (0, self.max_len - wav.shape[1]))
            wav = wav[:, : self.max_len]
            spec = self.db(self.mel(wav)).clamp(min=-80, max=80)
            spec = (spec + 80.0) / 160.0
            y = torch.tensor(encode_single_label(row["primary_label"], label_to_idx), dtype=torch.float32)
            return spec, y

    class TinyAudioCNN(nn.Module):
        def __init__(self, n_classes: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv2d(1, 16, kernel_size=3, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(16, 32, kernel_size=3, padding=1),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.AdaptiveAvgPool2d((1, 1)),
            )
            self.head = nn.Linear(64, n_classes)

        def forward(self, x):
            x = self.net(x).flatten(1)
            return self.head(x)

    train_loader = DataLoader(AudioSmokeDataset(smoke_train), batch_size=8, shuffle=True, num_workers=2)
    val_loader = DataLoader(AudioSmokeDataset(smoke_val), batch_size=8, shuffle=False, num_workers=2)

    model = TinyAudioCNN(len(label_cols)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for batch_idx, (x, y) in enumerate(train_loader):
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        logger.info("Neural smoke train batch=%s loss=%.6f x_shape=%s", batch_idx, float(loss.item()), tuple(x.shape))
        if batch_idx >= 2:
            break

    model.eval()
    preds = []
    truth = []
    with torch.no_grad():
        for x, y in val_loader:
            logits = model(x.to(device))
            preds.append(torch.sigmoid(logits).cpu().numpy())
            truth.append(y.numpy())
    if preds:
        y_pred = np.concatenate(preds)
        y_true = np.concatenate(truth)
        logger.info("Neural smoke val shapes y_true=%s y_pred=%s", y_true.shape, y_pred.shape)
        try:
            logger.info("Neural smoke random-quality macro_auc=%.6f", kaggle_macro_auc_skip_empty(y_true, y_pred))
        except Exception as exc:
            logger.warning("Neural smoke metric failed: %s", repr(exc))
else:
    logger.info("Skipping neural smoke test. Set RUN_NEURAL_SMOKE=True to enable.")



## Stage 3b Optional: Neural Baseline Training

This is the first reportable PyTorch baseline for experiment A2. It uses the
original `train_audio` split, mel-spectrogram inputs, a compact CNN, and the
same validation metric as the BirdNET ML baseline.



In [ ]:
if RUN_NEURAL_BASELINE:
    ensure_neural_torch_runtime()
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset

    try:
        import torchaudio
    except Exception as exc:
        raise RuntimeError("torchaudio is required for neural baseline training") from exc

    logger.info("torch=%s cuda_available=%s", torch.__version__, torch.cuda.is_available())
    device = choose_torch_device(torch, nn)

    def neural_row_labels(row: pd.Series) -> List[str]:
        labels = [str(row["primary_label"])]
        if NEURAL_USE_SECONDARY_LABELS and "secondary_labels" in row:
            labels.extend(parse_secondary_labels(row["secondary_labels"]))
        return sorted(set(label for label in labels if label in label_to_idx))

    def balanced_neural_sample(rows: pd.DataFrame, max_rows: Optional[int], seed: int) -> pd.DataFrame:
        rows = rows.copy()
        if max_rows is None or len(rows) <= int(max_rows):
            return rows.sample(frac=1.0, random_state=seed).reset_index(drop=True)

        max_rows = int(max_rows)
        per_class_first = rows.groupby("primary_label", group_keys=False).sample(n=1, random_state=seed)
        remainder = rows.drop(index=per_class_first.index, errors="ignore")
        fill_n = max(0, max_rows - len(per_class_first))
        if fill_n > 0:
            fill = remainder.sample(n=min(fill_n, len(remainder)), random_state=seed)
            selected = pd.concat([per_class_first, fill], axis=0)
        else:
            selected = per_class_first.sample(n=max_rows, random_state=seed)
        return selected.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    def discover_synthetic_audio_rows(max_rows: Optional[int], seed: int) -> pd.DataFrame:
        if not NEURAL_INCLUDE_SYNTHETIC:
            return pd.DataFrame()
        if not Path("/kaggle/input").exists():
            logger.warning("NEURAL_INCLUDE_SYNTHETIC=True but /kaggle/input is not available")
            return pd.DataFrame()

        manifest_files = sorted(Path("/kaggle/input").rglob("synthetic_generation_manifest.csv"))
        rows = []
        for manifest_path in manifest_files:
            chunk_root = manifest_path.parent
            manifest = pd.read_csv(manifest_path)
            for _, row in manifest.iterrows():
                if str(row.get("filter_status")) != "generated_pending_birdnet_filter":
                    continue
                rel_path = str(row.get("relative_audio_path"))
                audio_path = chunk_root / rel_path
                if not audio_path.exists():
                    continue
                rows.append(
                    {
                        "primary_label": str(row["primary_label"]),
                        "secondary_labels": "[]",
                        "filename": rel_path,
                        "split": "train",
                        "split_reason": "synthetic_audioldm2_train_only",
                        "audio_path": str(audio_path),
                        "source_kind": "synthetic",
                        "synthetic_id": str(row["synthetic_id"]),
                    }
                )

        if not rows:
            logger.warning("No synthetic audio rows discovered under /kaggle/input")
            return pd.DataFrame()

        synthetic_df = pd.DataFrame(rows).drop_duplicates("synthetic_id")
        synthetic_df = synthetic_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        if max_rows is not None and len(synthetic_df) > int(max_rows):
            synthetic_df = synthetic_df.head(int(max_rows)).copy()
        logger.info("Discovered synthetic neural train rows: %s", len(synthetic_df))
        return synthetic_df

    neural_train_rows = balanced_neural_sample(
        train_audio_split_df[train_audio_split_df["split"] == "train"],
        NEURAL_TRAIN_MAX_ROWS,
        SEED,
    )
    neural_train_rows = neural_train_rows.copy()
    neural_train_rows["source_kind"] = "real_train_audio"

    synthetic_neural_rows = discover_synthetic_audio_rows(NEURAL_SYNTHETIC_MAX_ROWS, SEED + 2)
    if not synthetic_neural_rows.empty:
        neural_train_rows = pd.concat([neural_train_rows, synthetic_neural_rows], axis=0, ignore_index=True)
        neural_train_rows = neural_train_rows.sample(frac=1.0, random_state=SEED + 3).reset_index(drop=True)

    neural_val_rows = balanced_neural_sample(
        train_audio_split_df[train_audio_split_df["split"] == "val"],
        NEURAL_VAL_MAX_ROWS,
        SEED + 1,
    )
    neural_val_rows = neural_val_rows.copy()
    neural_val_rows["source_kind"] = "real_train_audio"
    neural_train_rows.to_csv(WORK_DIR / "neural_baseline_train_manifest.csv", index=False)
    neural_val_rows.to_csv(WORK_DIR / "neural_baseline_val_manifest.csv", index=False)

    class AudioBaselineDataset(Dataset):
        def __init__(self, rows: pd.DataFrame, training: bool):
            self.rows = rows.reset_index(drop=True)
            self.training = training
            self.max_len = int(NEURAL_SECONDS * NEURAL_TARGET_SR)
            self.mel = torchaudio.transforms.MelSpectrogram(
                sample_rate=NEURAL_TARGET_SR,
                n_fft=1024,
                hop_length=320,
                n_mels=128,
                f_min=20,
                f_max=15000,
                power=2.0,
            )
            self.db = torchaudio.transforms.AmplitudeToDB(stype="power")
            self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=16)
            self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=24)

        def __len__(self):
            return len(self.rows)

        def crop_or_pad(self, wav: torch.Tensor) -> torch.Tensor:
            current = wav.shape[1]
            if current < self.max_len:
                return torch.nn.functional.pad(wav, (0, self.max_len - current))
            if current == self.max_len:
                return wav
            if self.training:
                start = int(torch.randint(0, current - self.max_len + 1, (1,)).item())
            else:
                start = max(0, (current - self.max_len) // 2)
            return wav[:, start : start + self.max_len]

        def __getitem__(self, idx):
            row = self.rows.iloc[idx]
            if "audio_path" in row and pd.notna(row.get("audio_path")) and str(row.get("audio_path")):
                path = Path(str(row["audio_path"]))
            else:
                path = train_audio_path(row["filename"])
            wav, sr = torchaudio.load(str(path))
            wav = wav.mean(dim=0, keepdim=True)
            if sr != NEURAL_TARGET_SR:
                wav = torchaudio.functional.resample(wav, sr, NEURAL_TARGET_SR)
            wav = self.crop_or_pad(wav)
            if self.training:
                gain = 0.8 + 0.4 * torch.rand(1).item()
                wav = wav * gain
                if torch.rand(1).item() < 0.35:
                    wav = wav + torch.randn_like(wav) * 0.002
            spec = self.db(self.mel(wav)).clamp(min=-80, max=80)
            spec = (spec + 80.0) / 160.0
            if self.training:
                spec = self.freq_mask(spec)
                spec = self.time_mask(spec)
            y = torch.tensor(encode_multi_label(neural_row_labels(row), label_to_idx), dtype=torch.float32)
            return spec, y

    class CompactAudioCNN(nn.Module):
        def __init__(self, n_classes: int):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.SiLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(64),
                nn.SiLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(128),
                nn.SiLU(inplace=True),
                nn.MaxPool2d(2),
                nn.Conv2d(128, 192, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(192),
                nn.SiLU(inplace=True),
                nn.AdaptiveAvgPool2d((1, 1)),
            )
            self.head = nn.Sequential(
                nn.Dropout(0.25),
                nn.Linear(192, n_classes),
            )

        def forward(self, x):
            return self.head(self.features(x).flatten(1))

    y_train_matrix = np.stack(
        [encode_multi_label(neural_row_labels(row), label_to_idx) for _, row in neural_train_rows.iterrows()]
    )
    positives = y_train_matrix.sum(axis=0)
    pos_weight = (len(y_train_matrix) - positives) / np.maximum(positives, 1)
    pos_weight = np.clip(pos_weight, 1.0, 25.0).astype(np.float32)

    train_loader = DataLoader(
        AudioBaselineDataset(neural_train_rows, training=True),
        batch_size=NEURAL_BATCH_SIZE,
        shuffle=True,
        num_workers=NEURAL_NUM_WORKERS,
        pin_memory=device.type == "cuda",
        drop_last=False,
    )
    val_loader = DataLoader(
        AudioBaselineDataset(neural_val_rows, training=False),
        batch_size=NEURAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NEURAL_NUM_WORKERS,
        pin_memory=device.type == "cuda",
        drop_last=False,
    )

    model = CompactAudioCNN(len(label_cols)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=NEURAL_LR, weight_decay=NEURAL_WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
    scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    history = []
    best_auc = -1.0
    best_epoch = -1
    best_pred = None
    best_true = None
    best_filenames = None

    logger.info(
        "Neural baseline config train_rows=%s val_rows=%s epochs=%s batch_size=%s seconds=%s target_sr=%s",
        len(neural_train_rows),
        len(neural_val_rows),
        NEURAL_EPOCHS,
        NEURAL_BATCH_SIZE,
        NEURAL_SECONDS,
        NEURAL_TARGET_SR,
    )

    for epoch in range(1, NEURAL_EPOCHS + 1):
        model.train()
        losses = []
        start = time.time()
        for batch_idx, (x, y) in enumerate(train_loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=device.type == "cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu().item()))
            if batch_idx % 25 == 0:
                logger.info("Neural epoch=%s batch=%s loss=%.6f", epoch, batch_idx, losses[-1])

        model.eval()
        preds = []
        truth = []
        filenames = []
        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(val_loader):
                x = x.to(device, non_blocking=True)
                logits = model(x)
                preds.append(torch.sigmoid(logits).cpu().numpy())
                truth.append(y.numpy())
                start_i = batch_idx * NEURAL_BATCH_SIZE
                end_i = start_i + len(y)
                filenames.extend(neural_val_rows.iloc[start_i:end_i]["filename"].tolist())

        y_pred = np.concatenate(preds, axis=0)
        y_true = np.concatenate(truth, axis=0)
        val_auc = kaggle_macro_auc_skip_empty(y_true, y_pred)
        elapsed = time.time() - start
        row = {
            "epoch": int(epoch),
            "train_loss": float(np.mean(losses)) if losses else float("nan"),
            "val_macro_auc_skip_empty": float(val_auc),
            "elapsed_seconds": float(elapsed),
        }
        history.append(row)
        logger.info("Neural epoch summary: %s", row)
        if val_auc > best_auc:
            best_auc = float(val_auc)
            best_epoch = int(epoch)
            best_pred = y_pred
            best_true = y_true
            best_filenames = filenames
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "label_cols": label_cols,
                    "config": {
                        "seconds": NEURAL_SECONDS,
                        "target_sr": NEURAL_TARGET_SR,
                        "n_mels": 128,
                        "model": "CompactAudioCNN",
                    },
                    "best_epoch": best_epoch,
                    "best_val_macro_auc_skip_empty": best_auc,
                },
                WORK_DIR / "neural_baseline_best.pt",
            )

    history_df = pd.DataFrame(history)
    history_df.to_csv(WORK_DIR / "neural_baseline_history.csv", index=False)

    if best_pred is not None:
        val_pred_df = pd.DataFrame(best_pred, columns=label_cols)
        val_pred_df.insert(0, "filename", best_filenames)
        val_pred_df.to_csv(WORK_DIR / "neural_baseline_val_predictions.csv", index=False)

    neural_summary = {
        "model": "CompactAudioCNN",
        "train_rows": int(len(neural_train_rows)),
        "val_rows": int(len(neural_val_rows)),
        "epochs": int(NEURAL_EPOCHS),
        "batch_size": int(NEURAL_BATCH_SIZE),
        "seconds": float(NEURAL_SECONDS),
        "target_sr": int(NEURAL_TARGET_SR),
        "use_secondary_labels": bool(NEURAL_USE_SECONDARY_LABELS),
        "include_synthetic": bool(NEURAL_INCLUDE_SYNTHETIC),
        "synthetic_train_rows": int((neural_train_rows.get("source_kind", pd.Series(dtype=str)) == "synthetic").sum()),
        "train_positive_classes": int((y_train_matrix.sum(axis=0) > 0).sum()),
        "val_positive_classes": int(best_true.sum(axis=0).astype(bool).sum()) if best_true is not None else 0,
        "best_epoch": int(best_epoch),
        "best_val_macro_auc_skip_empty": float(best_auc),
        "history_path": str(WORK_DIR / "neural_baseline_history.csv"),
        "checkpoint_path": str(WORK_DIR / "neural_baseline_best.pt"),
    }
    log_json("Neural baseline summary", neural_summary)
    (WORK_DIR / "neural_baseline_summary.json").write_text(
        json.dumps(neural_summary, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    display(pd.DataFrame([neural_summary]).T.rename(columns={0: "value"}))
else:
    logger.info("Skipping neural baseline. Set RUN_NEURAL_BASELINE=True to enable.")



## Outputs to Send Back

After Stage 0, send:
- the visible notebook output for EDA and split summary
- `/kaggle/working/a2_run.log`
- `/kaggle/working/eda_summary.json`
- `/kaggle/working/split_summary.json`
- `/kaggle/working/audio_path_summary.json`

After BirdNET smoke, also send:
- `/kaggle/working/birdnet_smoke_manifest.csv`
- first rows and column names of `/kaggle/working/birdnet_smoke_embeddings.csv`
- any error text in `a2_run.log`

After BirdNET ML baseline, send:
- `/kaggle/working/birdnet_ml_baseline_summary.json`
- `/kaggle/working/birdnet_ml_val_predictions.csv`

After Neural baseline, send:
- `/kaggle/working/neural_baseline_summary.json`
- `/kaggle/working/neural_baseline_history.csv`
- `/kaggle/working/neural_baseline_val_predictions.csv`



In [ ]:
output_files = sorted(str(p) for p in WORK_DIR.glob("*") if p.is_file())
logger.info("Files currently in WORK_DIR:\n%s", "\n".join(output_files))
display(pd.DataFrame({"output_file": output_files}))

logger.info("Notebook stage complete.")
print(f"Done. Main log: {LOG_PATH}")
